一、导入相关库

In [17]:
import os
import sys
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
import alphalens as al
import matplotlib.pyplot as plt
from datetime import datetime
import calendar

# 配置文件
try:
    import config_local as config
except ImportError:
    import config

# 导入数据接口sdk
import zenidatasdk as zd
client = zd.Client(
    base_url = config.ZENI_URL,
    username = config.ZENI_USERNAME,
    password = config.ZENI_PASSWORD,
)

# 忽视警告信息
warnings.filterwarnings(action = 'ignore')

二、获取数据

In [18]:
# 历史回测区间
init_date = '2017-01-01'
start_date = '2018-01-01'
end_date = str(datetime.today().date())
index_symbol = rf"000852.XSHG"

# rf"000300.XSHG",  # 沪深300
# rf"000905.XSHG",  # 中证500
# rf"000852.XSHG",  # 中证1000
# rf"000016.XSHG",  # 上证50
# rf"399006.XSHE",  # 创业板

In [19]:
# 获取指数成分股数据据
index_weights_df = client.get_index_constituents_df(
    index_symbol = index_symbol,
    start_date = start_date,
    end_date = end_date
)
index_weights_df = index_weights_df.rename(columns = {"date": "datetime"})
symbols = index_weights_df["symbol"].unique().tolist()
index_weights_df

,datetime,index_symbol,update_date,symbol,weight,display_name
0,2018-01-02,000852.XSHG,2017-12-29,000010.XSHE,0.073,美丽生态
1,2018-01-02,000852.XSHG,2017-12-29,000011.XSHE,0.063,深物业A
2,2018-01-02,000852.XSHG,2017-12-29,000016.XSHE,0.159,深康佳A
3,2018-01-02,000852.XSHG,2017-12-29,000018.XSHE,0.103,神城A退
4,2018-01-02,000852.XSHG,2017-12-29,000029.XSHE,0.070,深深房A
...,...,...,...,...,...,...
2008995,2026-04-16,000852.XSHG,2026-03-31,688776.XSHG,0.055,国光电气
2008996,2026-04-16,000852.XSHG,2026-03-31,688779.XSHG,0.143,五矿新能
2008997,2026-04-16,000852.XSHG,2026-03-31,688789.XSHG,0.084,宏华数科
2008998,2026-04-16,000852.XSHG,2026-03-31,688798.XSHG,0.073,艾为电子


In [20]:
tmp_Dataset_path = rf"C:\Users\zy\Desktop\tmp_Dataset"
minute_bar_Dataset_path = os.path.join(tmp_Dataset_path, "minute_bar") # 分钟数据本地路径
os.makedirs(minute_bar_Dataset_path, exist_ok = True)

index_weights_path = os.path.join(minute_bar_Dataset_path, index_symbol) # 指数成分股
os.makedirs(index_weights_path, exist_ok = True)

In [21]:
def generate_monthly_ranges_dict(start_date, end_date):
    """生成按月切割的日期区间字典，格式为 {YYMM: (月初, 月末)}"""
    start = datetime.strptime(start_date, '%Y-%m-%d').date()
    end = datetime.strptime(end_date, '%Y-%m-%d').date()

    date_dict = {}
    current = start

    while True:
        month_start = current.replace(day=1)
        _, last_day = calendar.monthrange(current.year, current.month)
        month_end = current.replace(day=last_day)

        if month_end > end:  # 不足一个月舍弃
            break

        # 键格式：YYMM（如 2603 表示 2026年03月）
        key = f"{str(current.year)[-2:]}{current.month:02d}"
        date_dict[key] = (month_start.strftime('%Y-%m-%d'), month_end.strftime('%Y-%m-%d'))

        # 移到下个月
        if current.month == 12:
            current = current.replace(year=current.year + 1, month=1, day=1)
        else:
            current = current.replace(month=current.month + 1, day=1)

    return date_dict


date_tuple_list = generate_monthly_ranges_dict(init_date, end_date)
date_tuple_list

{'1701': ('2017-01-01', '2017-01-31'),
 '1702': ('2017-02-01', '2017-02-28'),
 '1703': ('2017-03-01', '2017-03-31'),
 '1704': ('2017-04-01', '2017-04-30'),
 '1705': ('2017-05-01', '2017-05-31'),
 '1706': ('2017-06-01', '2017-06-30'),
 '1707': ('2017-07-01', '2017-07-31'),
 '1708': ('2017-08-01', '2017-08-31'),
 '1709': ('2017-09-01', '2017-09-30'),
 '1710': ('2017-10-01', '2017-10-31'),
 '1711': ('2017-11-01', '2017-11-30'),
 '1712': ('2017-12-01', '2017-12-31'),
 '1801': ('2018-01-01', '2018-01-31'),
 '1802': ('2018-02-01', '2018-02-28'),
 '1803': ('2018-03-01', '2018-03-31'),
 '1804': ('2018-04-01', '2018-04-30'),
 '1805': ('2018-05-01', '2018-05-31'),
 '1806': ('2018-06-01', '2018-06-30'),
 '1807': ('2018-07-01', '2018-07-31'),
 '1808': ('2018-08-01', '2018-08-31'),
 '1809': ('2018-09-01', '2018-09-30'),
 '1810': ('2018-10-01', '2018-10-31'),
 '1811': ('2018-11-01', '2018-11-30'),
 '1812': ('2018-12-01', '2018-12-31'),
 '1901': ('2019-01-01', '2019-01-31'),
 '1902': ('2019-02-01', '

In [25]:
big_minute_bar_dict = {}

from tqdm import tqdm
for month, start_end_tuple in tqdm(
        date_tuple_list.items(), desc = "处理月度数据"
):
    minute_bar_monthly_path = os.path.join(index_weights_path, rf"{month}.parquet")
    if os.path.exists(minute_bar_monthly_path):
        # bars_1m_df = pd.read_csv(minute_bar_monthly_path)
        pass
    else:
        bars_1m_df = client.get_kline_df(
            symbol = symbols,
            start_date = start_end_tuple[0],
            end_date = start_end_tuple[1],
            frequency = "1m",
            adjust_type = "post",
            market = "cn_stock",
        )
        bars_1m_df.to_csv(minute_bar_monthly_path, index = False)

    # big_minute_bar_dict[month] = bars_1m_df

处理月度数据: 100%|██████████| 111/111 [1:23:07<00:00, 44.93s/it]


In [24]:
import gc; gc.collect()

2555